# PMAP-Net training

Run the setup cell in Colab to clone the GitHub repository into `/content` and mount Google Drive. The default run trains on the saved 4×4 BPSK data at 5 dB. It reads `layer_0.csv` through `layer_3.csv` from `MyDrive/EE_MIMO_PROJECT/02_colab_output/rayleigh_4_by_4_BPSK/5dB/`.

Each run saves the model checkpoint and per-layer loss figure in a timestamped Drive folder.

In [ ]:
# @title Clone the GitHub repository and mount Drive
from pathlib import Path
import subprocess
import sys

from google.colab import drive

REPO_URL = "https://github.com/billlllllllly/EE_MIMO_PROJECT.git"
CODE_ROOT = Path("/content/EE_MIMO_PROJECT")
DRIVE_ROOT = Path("/content/drive/MyDrive/EE_MIMO_PROJECT")

if (CODE_ROOT / ".git").exists():
    origin = subprocess.check_output(
        ["git", "remote", "get-url", "origin"], cwd=CODE_ROOT, text=True
    ).strip()
    if origin != REPO_URL:
        raise RuntimeError(f"{CODE_ROOT} is a clone of {origin}. Set CODE_ROOT to a new directory.")
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(CODE_ROOT)],
        check=True,
    )

trainer_path = CODE_ROOT / "src/models/train_pmapnet.py"
if not trainer_path.is_file():
    raise FileNotFoundError(f"Could not find the trainer in the GitHub checkout at {trainer_path}.")

if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))
from src.models.train_pmapnet import main as train_pmapnet
if Path(train_pmapnet.__code__.co_filename).resolve() != trainer_path.resolve():
    raise RuntimeError("Restart the Colab runtime to load the trainer from the GitHub checkout.")

drive.mount("/content/drive")

In [ ]:
# @title Training settings
MODULATION = "bpsk"
NUM_TX = 4
HIDDEN_LAYERS = 2
EPOCHS = 100
BATCH_SIZE = 100
TRAIN_SAMPLES = 20_000
LEARNING_RATE = 0.005
SEED = 42
DEVICE = "auto"  # "auto", "cuda", or "cpu"

LAYER_DATA = "02_colab_output/rayleigh_4_by_4_BPSK/5dB"  # Relative to DRIVE_ROOT
OUTPUT_DIR = DRIVE_ROOT / "02_colab_output"

In [ ]:
# @title Train PMAP-Net
arguments = [
    "--modulation", MODULATION,
    "--num-tx", str(NUM_TX),
    "--hidden-layers", str(HIDDEN_LAYERS),
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--train-samples", str(TRAIN_SAMPLES),
    "--learning-rate", str(LEARNING_RATE),
    "--seed", str(SEED),
    "--output-dir", str(OUTPUT_DIR),
    "--layer-data", str(DRIVE_ROOT / LAYER_DATA),
]
if DEVICE != "auto":
    arguments.extend(["--device", DEVICE])

run_dir = train_pmapnet(arguments)

In [ ]:
# @title Show the loss curves and saved files
from IPython.display import Image, display

display(Image(filename=str(run_dir / "layer_loss.png")))
print(f"Saved in {run_dir}")
for checkpoint in sorted(run_dir.glob("pmapnet_*.pt")):
    print(checkpoint.name)